# Streamlit과 외부 라이브러리 연동하기

## 1. Streamlit - Folium 연동
#### 파이썬에서 성능 좋은 지도 시각화 라이브러리인 Folium을 Stramlit과 연동하기
- Streamlit 자체 map 함수보다 더 많은 설정이 필요하지만 그만큼 커스터마이징이 가능한 기능들을 다수 포함하고 있음
- `참조 사이트`: https://folium.streamlit.app/
- `깃허브`: https://github.com/randyzwitch/streamlit-folium

In [3]:
!pip install folium
!pip install streamlit_folium

In [7]:
%%writefile module/myApp24.py

import streamlit as st
import folium
import streamlit_folium as sf
import numpy as np

# 미니맵 출력 기능, 최대화 창 기능, 히트맵 출력 기능(마커가 많은 곳은 색상이 붉고 진하게 나옴)
from folium.plugins import MiniMap, Fullscreen, HeatMap

st.title('🗺️ Folium - Streamlit 연동')

# 1. 위도, 경도 입력 UI 생성
col1, col2 = st.columns([1, 1])
with col1:
    # 기본 위경도는 서울 시청 값 
    lat0 = st.number_input('위도 입력: ', value=37.55, format='%.2f', step=0.01)

with col2:
    lon0 = st.number_input('경도 입력: ', value=126.99, format='%.2f', step=0.01)

    
# 2. 지도 스타일 선택
tile_option = st.selectbox(
    '지도 스타일 선택',
    ['OpenStreetMap',
    'CartoDB positron',
    'CartoDB dark_matter',
    'Esri WorldStreetMap',
    'Esri WorldImagery']
)


# 3. 랜덤 마커 시드를 세션으로 관리
if 'seed' not in st.session_state:
    st.session_state['seed'] = 1  # 시드 초기값 숫자는 아무거나 사용해도 됨

if st.button('마커 시드 변경'):
    st.session_state['seed'] += 1  # 여기도 마찬가지. 숫자는 의미가 없음

st.write('현재 시드값:', st.session_state['seed'])


# 4-1. 랜덤 마커값(위도, 경도) 생성 함수 선언
@st.cache_data
def random_markers(n, lat, lon, seed):        # 마커개수, 위도, 경도, 시드값
    np.random.seed(seed)  # 랜덤시드값 고정
    # 원하는 개수만큼 랜덤값을 추가해서 마커 좌표값 생성
    markers = [
        [lat+np.random.randn() / 20, lon+np.random.randn() / 20] for _ in range(n)
    ]

    return markers

# 4-2. 함수 실행하여 랜덤 마커 생성
markers = random_markers(50, lat0, lon0, st.session_state['seed'])
# # <출력 확인용>
# st.write(markers[:3])


# 5. folium 지도 객체 생성
    # location: 초기 중심 좌표값
    # zoom_start: 시작시 줌 배율
    # tiles: 지도 스타일 설정
myMap = folium.Map(location=[lat0, lon0],
                  zoom_start=11,
                  tiles=tile_option)


# 6-1. 마커를 묶어줄 피쳐그룹 객체 생성 (여러 마커들을 묶어서 관리하기 위함)
    # FeatureGroup: 지도 위 여러 요소들을 한 묶음(layer)으로 그룹화
    # add_to: 피쳐그룹 객체의 정보가 표시될 지도 객체 지정
marker_layer = folium.FeatureGroup(name='마커').add_to(myMap)

# 6-2. 여러개의 마커들을 랜덤하게 생성하여 피쳐그룹(marker_layer)에 반복해서 넣기
# location은 [lat, lon] 값 하나만 기대하는데 markers는 좌표 50개 묶음이라 for문 사
popup_cnt = 1
for lat, lon in markers:
    folium.Marker(
        location=[lat, lon],             # 마커가 표시될 위,경도
        popup=f'PopUp_{popup_cnt}',      # 마커 클릭시 출력 팝업 
        tooltip='클릭해보세요!',           # 마커에 마우스 오버시 출력 문구
        icon=folium.Icon(color='blue', icon='star'),  # icon: 마커 출력모양 설정
    ).add_to(marker_layer)               # 해당 마커들을 피쳐그룹에 포함시키기
    popup_cnt += 1


# 7. 히트맵 출력 (마커가 집중된 곳일수록 붉은색)
HeatMap(markers, name='히트맵').add_to(myMap)

# 8. 미니맵, 풀스크린 버튼 설정
MiniMap().add_to(myMap)
Fullscreen().add_to(myMap)

# 9. layer 컨트롤 창 추가 (FeatureGroup이나 Heatmap으로 추가한 layer를 on,off 할 수 있는 창)
folium.LayerControl().add_to(myMap)

# 10. streamlit으로 지도 출력
sf.st_folium(myMap, width=700, height=500, key='main_map')

Overwriting module/myApp24.py


In [6]:
# 피처그룹에 따로 담아야 체크박스에서 선택할 수 있음 

import folium
import streamlit_folium as sf


m = folium.Map(location=[37.5665, 126.9780], zoom_start=12)

# 1. 치킨 바구니 만들고 마커 담기
chicken_group = folium.FeatureGroup(name='치킨집만 보기')
folium.Marker([37.5, 126.9], popup='치킨').add_to(chicken_group)
chicken_group.add_to(m) # 지도에 바구니 넣기

# 2. 피자 바구니 만들고 마커 담기
pizza_group = folium.FeatureGroup(name='피자집만 보기')
folium.Marker([37.5, 127.0], popup='피자').add_to(pizza_group)
pizza_group.add_to(m) # 지도에 바구니 넣기

# 3. 중요!! 스위치(레이어 컨트롤) 설치하기
# 이 한 줄이 있어야 비로소 지도 위에 체크박스가 생깁니다.
folium.LayerControl().add_to(m)

# 지도 보여줘!
sf.st_folium(m, width=700, height=500, key='main_map')

2026-02-03 09:40:27.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 09:40:27.357 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 09:40:27.358 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 09:40:27.361 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 09:40:27.524 
  command:

    streamlit run C:\Users\agnes\AppData\Roaming\Python\Python313\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-03 09:40:27.526 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-03 09:40:27.528 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


{'last_clicked': None,
 'last_object_clicked': None,
 'last_object_clicked_tooltip': None,
 'last_object_clicked_popup': None,
 'all_drawings': None,
 'last_active_drawing': None,
 'bounds': {'_southWest': {'lat': 37.5, 'lng': 126.9},
  '_northEast': {'lat': 37.5, 'lng': 127.0}},
 'zoom': 12,
 'last_circle_radius': None,
 'last_circle_polygon': None,
 'selected_layers': None}

## Streamlit - Option Menu 연동
#### Streamlit으로 완성된 메뉴 UI를 만들어주는 라이브러리
- `깃허브`: https://github.com/victoryhb/streamlit-option-menu
- `네이게이션 아이콘 참조 사이트`: https://icons.getbootstrap.com/
- `HEX값 확인 사이트`: https://htmlcolorcodes.com/

In [21]:
!pip install streamlit-option-menu

   ---------------------------------------- 0.0/829.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/829.3 kB ? eta -:--:--
   ---------------------------------------- 829.3/829.3 kB 4.7 MB/s  0:00:00


In [52]:
%%writefile module/myApp25.py

import streamlit as st
import pandas as pd
import numpy as np
from streamlit_option_menu import option_menu

# 페이지 기본 설정
st.set_page_config(
    layout='wide',
    page_title='AI Dashboard',
    page_icon='📉'
)

# === [사이드바 부분] ===
with st.sidebar:
    col1, col2 = st.columns([1, 5])
    
    # 로고 부분 (좌측)
    with col1:
        st.image("https://slack-imgs.com/?c=1&o1=ro&url=https%3A%2F%2Fcdn-icons-png.flaticon.com%2F512%2F2103%2F2103633.png", width=50)
    # 텍스트 부분 (우측)
    with col2:
        st.markdown('### AI Control Center')

    #사이드바에 option_menu UI 배치
    option_menu_side = option_menu(
        menu_title='메뉴 선택',                   # 메뉴 박스 이름
        menu_icon='cast',                        # 이름 앞에 붙는 아이콘
        options=['대시보드', '데이터 분석', '설정'],  # 메뉴 탭 명칭들
        icons=['speedometer2', 'bar-chart-line', 'gear'], # 탭 앞에 붙는 아이콘
        default_index=0,       # 앱 실행시 가장 먼저 선택되어 있을 메뉴의 인덱스 번호
        styles={
            # container : 메뉴 탭들을 감싸는 전체 공간
            # padding : 요소 내부의 여백 (현재는 0으로 여백 없으며, !important는 해당스타일 우선 적용)
            # background-color : 배경색 (컬러 HEX코드 값으로 설정 가능하며 컬러명 문자로도 가능)
            'container':{'padding':'0!important','background-color':'#fafafa'},
            # 아이콘 세부 사항 설정
            'icon': {'color':'orange', 'font-size':'18px'},
            # nav-link: 메뉴 탭 내부 관련 설정
            # text-alignL 텍스트 정렬
            # margin: 메뉴 탭 가싸고 있는 박스와 텍스트 사이 공간
            # --hover-color: 메뉴에 마우스 오버시 변경되는 색상 (#eee 옅은 회색)
            'nav-link': {'font-size':'16px', 'text-align': 'left', 'margin':'0px',
                        '--hover-color': '#eee'},
            # nav-link-selected: 메뉴 탭이 선택되었을 때 설정 (#02ab21은 진한 녹색)
            'nav-link-selected': {'background-color':'#02ab21'}
        }
    )

    st.info(f'현재 접속: {option_menu_side} 모드')


# === [메인 부분] ===
# 1) 대시보드 선택 시
if option_menu_side == '대시보드':
    st.title('🚀대시보드')

    option_menu_main = option_menu(
        menu_title=None, # 메뉴 타이틀 숨기기 
        options=['요약 보기', '실시간 트래픽', '보고서 다운로드'],
        icons=['card-checklist','activity', 'cloud-download'],
        default_index=0,
        orientation='horizontal', # 메뉴 탭 출력 형태를 수평으로 설정 (수직은 vertical)
        styles={
            'container':{'padding':'1!important', 'background-color':'#E8E8E8'},
            'icon':{"color": "#000",'font-size':'15px'},
            'nav-link':{'font-size':'15px', "text-align": "center", 'margin':'1px',
                       '--hover-color': '#eee'},
            'nav-link-selected': {'background-color': 'red'}
        }
    )
    '---'
    # 메인 메뉴 요약보기 눌렀을 때
    if option_menu_main == '요약 보기':
        # KPI 지표 카드 예시
        col1, col2, col3 = st.columns(3)
        col1.metric('총 방문자', '12,450명', '▲ 15%')
        col2.metric('매출액', '$54,000', '▲8%')
        col3.metric('서버 상태', '정상', '99.9%')

        st.success('오늘의 주요 지표 현황입니다.')

    elif option_menu_main == '실시간 트래픽':
        st.subheader('📊실시간 접속 현황')
        # 더미 차트 생성
        chart_data = pd.DataFrame(np.random.randn(20,3),
                                 columns=['A', 'B', 'C'])
        st.line_chart(chart_data)

    elif option_menu_main == '보고서 다운로드':
        st.warning('🔒 관리자 권한이 필요합니다.')
        with st.empty(): 
            if st.button('권한 요청하기'):
                st.success('권한이 요청되었습니다.')

# 2) '데이터 분석' 선택 시
if option_menu_side == '데이터 분석':
    st.title('📉데이터 탐색기(EDA)')
    tab1, tab2 = st.tabs(['📁파일 업로드', '🔎데이터 미리보기'])
    with tab1:
        st.file_uploader('분석할 CSV 파일을 업로드하세요~!', type=['csv', 'xlsx'])
    with tab2:
        st.write('업로드된 데이터가 없습니다. 아래는 샘플 데이터입니다~~')
        st.dataframe(pd.DataFrame({'모델':['A', 'B'], '정확도':[0.95, 0.88]}))

# 3) '설정' 선택 시
if option_menu_side == '설정':
    st.title('⚙️환경설정')
    # toggle: on, off 설정 위젯
    st.toggle('다크 모드 적용')
    # value=True : 처음부터 on 상태
    st.toggle('알림 받기', value=True)

    st.markdown('### 사용자 정보')
    # type='password': 입력값이 보이지 않게 설정
    st.text_input('Password 입력', type='password')

    if st.button('저장하기'):
        # toast: 잠깐 떴다가 사라지는 인스턴ㄴ트 알림 메시지
        st.toast('설정이 저장되었습니다!', icon='✅')

Overwriting module/myApp25.py


## Streamlit - Lottie 연동
- lottie는 JSON포맷의 텍스트로 된 파일로 그림(애니메이션)을 그려주는 안내서 역할 (이를 바탕으로 웹 브라우저가 실시간으로 애니메이션을 그려서 출력해줌)
- `홈페이지 무료 애니메이션 리스트` : https://lottiefiles.com/featured-free-animations
- **애니메이션 주소 가져오기**: 사이트에서 원하는 애니메이션 클릭- 'Asset Embed' 탭 클릭 - 'save to ~~' 클릭(처음엔 save, 다음에는 open으로 뜸) - Handoff 탭 클릭 - 'Asset link format' 눌러서 Json 형태로 선택 - 'Asset link' 복사하기 클릭

In [53]:
!pip install streamlit-lottie

   ---------------------------------------- 0.0/802.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/802.4 kB ? eta -:--:--
   ---------------------------------------- 802.4/802.4 kB 6.3 MB/s  0:00:00


In [70]:
%%writefile module/myApp26.py
import streamlit as st
import requests as req
import time
from streamlit_lottie import st_lottie

# 페이지 기본설정
st.set_page_config(
    page_title='2026 AI Vision',
    page_icon='🚀',
    layout='wide'
)

# lottie 로드 함수 (캐시 적용)
# url 주소에 애니메이션 컨텐츠의 json 주소를 입력하고 출력도 json으로 출력 
@st.cache_data
def load_lottie_json(url):
    res = req.get(url)
    # status_code: <Response [200]>에서 []안의 수치값 확인
    if res.status_code != 200:
        return st.error('통신 에러 발생')
    return res.json()

# 애니메이션 2개에 대해 함수 실행
lottie1 = load_lottie_json(
    "https://lottie.host/b2028fe9-d806-4a8a-8319-7da059ca6504/f10kr05PrR.json"
)
lottie2 = load_lottie_json(
    "https://lottie.host/df90f25b-6928-4a95-9c94-8821d541778c/uhH0uOmErS.json"
)

# 상단 메인 배너
with st.container():
    col1, col2 = st.columns([3, 2])

    # 텍스트 파트 (좌)
    with col1:
        st.title('AI Assistant Service 🤖')
        st.markdown('''
        ### 당신의 업무 효율을 극대화하세요.
        AI 기반의 자동화 솔루션으로 반복 업무는 줄이고,
        **창의적인 일**에 더 집중할 수 있도록 도와드립니다.
        ''')
    # 버튼에 스타일링 넣기
     # type='primary': 중요 버튼(빨간색 배경)으로 시각화
    if st.button('무료 체험 시작하기', type='primary'):
        st.toast('환영합니다. 서비스 체험을 시작합니다. 🎉')
        
    # 애니메이션 파트 (우)
    with col2:
        if lottie1:
            # height: 애니 높이 (지정하지 않으면 원본 비율로 출력)
            # width: 너비 (지정하지 않으면 할당된 공간에 맞춰 출력)
            st_lottie(lottie1, height=300, key='a')

"---"

# 대시보드 및 목표 설정
st.subheader('📊2026 프로젝트 현황')

col1, col2, col3 = st.columns(3)
col1.metric('현재 사용자', '1,240명', '120명 증가')
col2.metric('AI 모델 정확도', '98.5%', '0.5%')
col3.metric('서버 가동률', '99.9%', '안정적')

# unsafe_allow_html=True : markdown에 HTML 코드 적용 설정
st.markdown('오잉<br>엥', unsafe_allow_html=True) # <br>은 개행 태그

with st.container():
    col1, col2 = st.columns([1, 2])

    # 애니메이션 파트 (좌)
    with col1:
        if lottie2:
            st_lottie(lottie2, height=250, key='b')
    # 텍스트 및 위젝 파트 (우)
    with col2:
        st.markdown('### 🚀2026년, 새로운 도약을 준비하세요!')
        st.write('여러분의 목표를 입력해주시면 AI가 맞춤형 로드맵을 제안해드립니다.')

        user_goal = st.text_input('올해 가장 이루고 싶은 목표는???',
                                  # placeholder: 입력 전 출력되는 옅은 회색 텍스트
                                 placeholder='예: 파이썬 마스터하기, 취업하기, 다이어트 성공하기')

        if st.button('목표 등록'):
            if user_goal:
                # 2초 대기 (현재는 AI는 없고 그냥분석하는 척하기 ^_^)
                with st.spinner('AI가 목표를 분석 중입니다...'):
                    time.sleep(2)
                st.success(f'멋진 목표네요! {user_goal} 달성을 응원합니다!🎁')
                st.balloons()
            else:
                st.warning('목표를 입력해주세요')

'---'
st.markdown('2026 AI Instructor. All rights reserved. | Powered by Streamlit & Lottie')

Overwriting module/myApp26.py


# 혼자 해보기

In [107]:
%%writefile module/practice.py

import streamlit as st
import requests as req
import time
from streamlit_lottie import st_lottie
from streamlit_option_menu import option_menu
import folium
import streamlit_folium as sf


############### 페이지 기본 설정 ###########
st.set_page_config(
    page_title='일단 만드는 중',
    page_icon='🔨',
    layout='wide'
)


################ 사이드바 설정 #############
with st.sidebar:
    col1, col2 = st.columns([10, 5])
    
    # 로고 부분 (좌측)
    with col1:
        st.image("https://cdn.imweb.me/upload/S20210809c06cc49e8b65a/e80edc05fd91a.png",)
    # 텍스트 부분 (우측)
    with col2:
        st.markdown('### 지금 **집** 가고 싶은 사람?')

    st.radio('진짜 솔직히 대답하기', ['저요', '나', '모두 같은 마음'])


    option_menu_side = option_menu(
        menu_title='효율적으로 집 가는 법',      
        menu_icon='house-fill',        
        options=['걸어가기', '대중교통 이용하기', '순간이동'], 
        icons=['person-walking', 'bus-front', 'arrow-up-circle-fill'], 
        default_index=0,   
        styles={
            'container':{'padding':'2!important','background-color':'#fafafa'},
            'icon': {'color':'black', 'font-size':'18px'},
            'nav-link': {'font-size':'15px', 'text-align': 'left', 'margin':'1px',
                        '--hover-color': '#eee'},
            'nav-link-selected': {'background-color':'gray'}
        }
    )

    st.info(f'현재 접속: {option_menu_side} 모드')

if option_menu_side == '걸어가기':
    st.title('🚶‍➡️뚜벅뚜벅 걸어가기🚶')

    option_menu_main = option_menu(
        menu_title=None, # 메뉴 타이틀 숨기기 
        options=['집 찾기', '실시간 트래픽'],
        icons=['card-checklist','activity'],
        default_index=0,
        orientation='horizontal', # 메뉴 탭 출력 형태를 수평으로 설정 (수직은 vertical)
        styles={
            'container':{'padding':'1!important', 'background-color':'#E8E8E8'},
            'icon':{"color": "#000",'font-size':'15px'},
            'nav-link':{'font-size':'15px', "text-align": "center", 'margin':'1px',
                       '--hover-color': '#eee'},
            'nav-link-selected': {'background-color': 'red'}
        }
    )
    if option_menu_main == '집 찾기':
        # 맵 넣어주기
        myHome = folium.Map(location=[37.5665, 126.9780], zoom_start=13)
        # 경로 (이미 계산된 좌표라고 가정)
        route = [
            [37.5665, 126.9780],
            [37.5651, 126.9895],
            [37.5700, 127.0000]
        ]
        # 경로 그리기
        folium.PolyLine(route, color="blue", weight=5).add_to(myHome)
    sf.st_folium(myHome, width=700, height=500, key='map')

# 대중교통
if option_menu_side == '대중교통 이용하기':
    st.title('🚌')
    # 버스, 지하철 고르기
    # 지역 고르기
    # 실시간 버스 노선, 지하철 노선 보여주기

if option_menu_side == '순간이동':
    st.title('☝️순간이동 가보자고')


    # 이미지 가져오기
    @st.cache_data
    def load_lottie_json(url):
        res = req.get(url)
        # status_code: <Response [200]>에서 []안의 수치값 확인
        if res.status_code != 200:
            return st.error('통신 에러 발생')
        return res.json()
    
    # 애니메이션 2개에 대해 함수 실행
    lottie2 = load_lottie_json(
        "https://lottie.host/9a496984-15ab-4bda-99dd-345087743117/gnkgEi2Vv3.json"
    )
    
    with st.container():
        col1, col2 = st.columns(2)
        
        with col1:
            if lottie2:
                st_lottie(lottie2, height=250, key='b')
        with col2:
            st.markdown('## 오늘도 고생하셨습니다!')
            st.write('지금 당장 당신의 집으로 향하세요! ☝️🏠')
    
            if st.button('집 가는 버튼', type='primary'):
                with st.spinner('향하는 중...'):
                    time.sleep(2)
                st.warning('그런 건 없습니다. 길 미끄러우니 조심히 가세요..')

'---'
st.caption('ⓒ 2026 How to go home | Powered by Streamlit & Lottie')

Overwriting module/practice.py
